# W2 Lab — Prompting and Reasoning: Chain-of-Thought and Examples

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week02/W2_lab_prompting.ipynb)

**Goal.** By the end of this lab you can make a model more accurate by letting it think
out loud (chain-of-thought), teach it a format and tone by showing examples instead of
writing instructions (few-shot), and verify that a prompt change actually helped by
scoring it on a fixed evalset with code.

This week's theory: reasoning accuracy depends on what the prompt makes the model
write before the answer. **Chain-of-thought (CoT)** = instructing the model to write
its intermediate steps first (Wei et al., 2022). **Few-shot prompting** = placing
worked examples in the prompt for the model to imitate.

The path: setup (Section 1) → thinking step by step, two observations (Section 2) → a
code-graded evaluation: baseline prompt → format fix → your CoT prompt, target 11 of
12 (Section 3) → few-shot examples and the email-classification exercise (Section 4)
→ self-consistency on the hardest item (Section 5) → completion (Section 6).

*Runtime:* Google Colab, top-to-bottom, ~80 minutes. Cells marked ✍️ contain fill-ins.

*Sources:* this lab adapts existing course material nearly as given. Sections 2 and 4:
Anthropic, *Prompt Engineering Interactive Tutorial*, chapters 6 (Thinking Step by
Step) and 7 (Using Examples) — the observation prompts, the email dataset, and the
grading loop are verbatim. Section 3: Anthropic, *Prompt Evaluations* course, lesson 3
(code-graded evals) — dataset, prompt sequence, and graders verbatim. Section 5
applies the procedure of Wang et al. (2022), *Self-Consistency*, to the Section 3
eval (no source lab exists for it). Adaptation is limited to the course API standard:
`aisuite` + a pasted key instead of the Anthropic SDK, and no assistant-prefill turns.

## 1. Setup

Same setup as W1: client library, key, the two helpers, one test call.

### 1.1 Installation

`aisuite` exposes multiple providers (OpenAI, Anthropic) behind one interface, so lab
code stays identical whichever provider your key belongs to.

*Do:* run the cell below (about 30 seconds, once per session).

In [ ]:
%pip install -q "aisuite[openai,anthropic]"

### 1.2 API key and model

An **API key** = the secret string that identifies your account to the provider and
bills usage to it (issuing steps: the API Setup guide on the course site). The key is
yours; do not share the notebook with the key still inside.

*Do:* replace `PASTE-YOUR-KEY-HERE` with your key and run the cell.

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"   # Anthropic accounts: MODEL = "anthropic:claude-haiku-4-5" and set ANTHROPIC_API_KEY instead

### 1.3 Client and helpers

The same two helpers as in W1: `chat` sends a full message list, `ask` wraps the
single-question case. The default temperature is 0.0 so that repeated runs are
comparable; Section 5 raises it deliberately.

*Do:* run the cell unchanged.

In [ ]:
import aisuite

client = aisuite.Client()

n_calls = 0
n_prompt_tokens = 0
n_completion_tokens = 0

def chat(messages, temperature=0.0, **kwargs):
    """Message list -> assistant reply text. Extra kwargs pass through to the provider."""
    global n_calls, n_prompt_tokens, n_completion_tokens
    response = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=temperature, **kwargs)
    n_calls += 1
    usage = getattr(response, "usage", None)
    if usage is not None:
        n_prompt_tokens += usage.prompt_tokens
        n_completion_tokens += usage.completion_tokens
    return response.choices[0].message.content

def ask(prompt, system=None, temperature=0.0, **kwargs):
    """Single question (optional system instruction) -> reply text."""
    messages = ([{"role": "system", "content": system}] if system else [])
    messages.append({"role": "user", "content": prompt})
    return chat(messages, temperature=temperature, **kwargs)

### 1.4 Verification

*Do:* run the cell and confirm the output is exactly `ready`.

In [ ]:
print(ask("Reply with exactly: ready"))

If the output is `ready`, key and billing work. Any error here is a setup problem, not
a code problem — recheck the API Setup guide before continuing.

## 2. Thinking Step by Step

Letting the model think out loud before answering sometimes makes it more accurate,
particularly on tasks with a trap or a constraint — and thinking only counts when it
is written out: asking the model to "think silently and output only the answer" does
no thinking at all (source: tutorial ch. 6). Two observations from the source, run
as given.

### 2.1 A sentiment that flips

In the review below, the second sentence undoes the first — the praise comes from
someone who has been living under a rock since 1900. Asked directly, the model tends
to take "unrelated" literally.

*Do:* run the cell and read the verdict.

In [ ]:
REVIEW_PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

print(ask(REVIEW_PROMPT))

The fix in the source: spell out the thinking the model should do — write the best
arguments for each side in tags, then answer.

*Do:* run the cell and compare the verdict with the previous one.

In [ ]:
THINK_FIRST_PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

print(ask(THINK_FIRST_PROMPT, system="You are a savvy reader of movie reviews."))

Writing out both sides let the model notice what the second sentence does to the
first. (The source also notes an ordering effect: the model is somewhat more likely
to pick the second of two options it argued, so the tag order matters.)

### 2.2 Recall under a constraint

The same device on a factual question the model tends to miss when answering cold.

*Do:* run both cells and compare — direct answer first, then with a brainstorming
step.

In [ ]:
print(ask("Name a famous movie starring an actor who was born in the year 1956."))

In [ ]:
print(ask("Name a famous movie starring an actor who was born in the year 1956. "
          "First brainstorm about some actors and their birth years in <brainstorm> tags, "
          "then give your answer."))

Two anecdotes are not evidence that step-by-step thinking helps in general. Whether it
does — and by how much — is a measurement question, and the next section sets up the
instrument.

## 3. A Code-Graded Evaluation

**Code-graded evaluation** = scoring a prompt by running it over a fixed test set with
known answers and grading the outputs with code (source: *Prompt Evaluations*,
lesson 3 — dataset and prompt sequence below are the source's own). The task: given a
statement about an animal, answer how many legs it has. Several statements are
deliberately tricky — a fox that lost a leg and regrew two, an octopus that lost two
and regrew three.

### 3.1 Eval set and first prompt

*Do:* run the two cells: the eval set, then the first prompt scored against it. Read
the outputs — note which are not bare numbers, and which numbers are wrong.

In [ ]:
eval_data = [
    {"animal_statement": "The animal is a human.", "golden_answer": "2"},
    {"animal_statement": "The animal is a snake.", "golden_answer": "0"},
    {"animal_statement": "The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.", "golden_answer": "5"},
    {"animal_statement": "The animal is a dog.", "golden_answer": "4"},
    {"animal_statement": "The animal is a cat with two extra legs.", "golden_answer": "6"},
    {"animal_statement": "The animal is an elephant.", "golden_answer": "4"},
    {"animal_statement": "The animal is a bird.", "golden_answer": "2"},
    {"animal_statement": "The animal is a fish.", "golden_answer": "0"},
    {"animal_statement": "The animal is a spider with two extra legs", "golden_answer": "10"},
    {"animal_statement": "The animal is an octopus.", "golden_answer": "8"},
    {"animal_statement": "The animal is an octopus that lost two legs and then regrew three legs.", "golden_answer": "9"},
    {"animal_statement": "The animal is a two-headed, eight-legged mythical creature.", "golden_answer": "8"},
]
print(len(eval_data), "items")

In [ ]:
def build_input_prompt(animal_statement):
    return f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{animal_statement}</animal_statement>

How many legs does the animal have? Please respond with a number"""

def grade_completion(output, golden_answer):
    return output.strip() == golden_answer

outputs = [ask(build_input_prompt(q["animal_statement"])) for q in eval_data]
for output, q in zip(outputs, eval_data):
    print(f"golden={q['golden_answer']:>2}  output={output!r}")
score_v1 = sum(grade_completion(o, q["golden_answer"]) for o, q in zip(outputs, eval_data))
print(f"\nscore: {score_v1}/12")

Two separate problems, as in the source run: some outputs are sentences rather than
bare numbers (a formatting failure — the right value graded wrong), and some numbers
are wrong on the tricky statements (a reasoning failure). The next two prompts fix
them one at a time.

### 3.2 Fixing the format

One added line: *"Respond only with a numeric digit, like 2 or 6, and nothing else."*

*Do:* run the cell; formatting failures should disappear, tricky items should still
miss.

In [ ]:
def build_input_prompt2(animal_statement):
    return f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{animal_statement}</animal_statement>

How many legs does the animal have? Respond only with a numeric digit, like 2 or 6, and nothing else."""

outputs2 = [ask(build_input_prompt2(q["animal_statement"])) for q in eval_data]
score_v2 = sum(grade_completion(o, q["golden_answer"]) for o, q in zip(outputs2, eval_data))
for output, q in zip(outputs2, eval_data):
    mark = "PASS" if grade_completion(output, q["golden_answer"]) else "FAIL"
    print(f"{mark}  golden={q['golden_answer']:>2}  output={output!r}")
print(f"\nscore: {score_v2}/12")

### 3.3 Chain of thought, graded ✍️ (core)

The remaining misses are reasoning failures, and Section 2's device is the candidate
fix. Whether it works is now checkable.

Write `build_input_prompt3`: keep the task statement, and instruct the model to
reason step by step inside `<thinking>` tags first, then put the final answer inside
`<answer>` tags — just the integer, nothing else. `extract_answer` (source verbatim)
pulls the number out of the tags for grading. Target: **at least 11 of 12**.

*Do:* fill in the prompt, run the check cell, and iterate on the wording until the
target is reached.

In [ ]:
### FILL IN (START) ###
def build_input_prompt3(animal_statement):
    return f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{animal_statement}</animal_statement>

How many legs does the animal have?"""
### FILL IN (END) ###

In [ ]:
import re

def extract_answer(text):
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    return match.group(1).strip() if match else None

outputs3 = [ask(build_input_prompt3(q["animal_statement"])) for q in eval_data]
extracted = [extract_answer(o) for o in outputs3]
cot_score = sum(e == q["golden_answer"] for e, q in zip(extracted, eval_data))
for e, q in zip(extracted, eval_data):
    mark = "PASS" if e == q["golden_answer"] else "FAIL"
    print(f"{mark}  golden={q['golden_answer']:>2}  extracted={e}  {q['animal_statement'][:58]}")
print(f"\nscore: {cot_score}/12   (target: >= 11)")

The source run reached 12/12 with this prompt shape. The point is not the animal
trivia: the eval made a prompting claim — "thinking step by step helps here" —
checkable, and the same three-step sequence (baseline → format fix → reasoning fix,
each scored) is how prompts are improved in practice.

## 4. Using Examples (Few-Shot)

**Few-shot prompting** = placing example question–answer pairs in the prompt; the
model imitates their format, tone, and procedure. "Zero-shot" / "one-shot" / "n-shot"
counts the examples. Often it is easier to show than to describe (source: tutorial
ch. 7).

### 4.1 The parent bot

The source's example: a bot that should answer children's questions. Asked cold, the
model answers like an encyclopedia.

*Do:* run both cells and compare the tone — no example, then one example.

In [ ]:
print(ask("Will Santa bring me presents on Christmas?"))

In [ ]:
PARENT_BOT_PROMPT = """Please complete the conversation by writing the next line, speaking as "A".
Q: Is the tooth fairy real?
A: Of course, sweetie. Wrap up your tooth and put it under your pillow tonight. There might be something waiting for you in the morning.
Q: Will Santa bring me presents on Christmas?"""

print(ask(PARENT_BOT_PROMPT))

One example fixed tone and length at once — no instruction described either.

### 4.2 Email classification by examples ✍️ (tutorial exercise 7.1)

The source exercise: classify customer emails into

- (A) Pre-sale question
- (B) Broken or defective item
- (C) Billing question
- (D) Other (please explain)

The grader checks the **last character** of the model's output against the correct
letter. Rewrite `PROMPT` so that few-shot examples of emails plus correctly formatted
answers make every output end with the right letter. (The source solution ends each
example answer with "The correct category is: X"; it also uses an assistant-prefill
turn, which is Anthropic-specific — put the examples inside the user prompt instead.)
Target: **4 of 4**.

*Do:* rewrite `PROMPT`, run the cell, and iterate until all four emails read `PASS`.

In [ ]:
import re

### FILL IN (START) ###
PROMPT = """Please classify this email as either green or blue: {email}"""
### FILL IN (END) ###

EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.",  # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?",  # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???",  # (C) Billing question
    "How did I get here I am not good with computer.  Halp.",  # (D) Other (please explain)
]
ANSWERS = [["B"], ["A", "D"], ["C"], ["D"]]

email_score = 0
for i, email in enumerate(EMAILS):
    response = ask(PROMPT.format(email=email))
    grade = any(bool(re.search(ans, response[-1])) for ans in ANSWERS[i])
    email_score += grade
    print(f"{'PASS' if grade else 'FAIL'}  expected {'/'.join(ANSWERS[i])}  got: ...{response[-40:]!r}")
print(f"\nscore: {email_score}/4")

The same classification task appears in the source's chapter 6 solved with
instructions alone; examples reach the same format with less instruction-writing, and
the two techniques compose — worked examples *of step-by-step solutions* are exactly
the few-shot CoT exemplars of Wei et al. (2022) (notes Ch. 2).

## 5. Self-Consistency

No source lab exists for this section; it applies the procedure of Wang et al. (2022)
directly to the Section 3 eval. **Self-consistency** = sample several reasoning paths
for the same question at nonzero temperature, then take the majority of the extracted
answers: wrong paths tend to scatter across different answers, correct paths agree.

*Do:* run the cell — five samples of the Section 3.3 prompt on the trickiest
statement at `t=1.0` — and compare the scattered samples with the majority.

In [ ]:
from collections import Counter

HARD = eval_data[2]   # the fox that lost a leg and regrew two

samples = [extract_answer(ask(build_input_prompt3(HARD["animal_statement"]), temperature=1.0))
           for _ in range(5)]
votes = [s for s in samples if s is not None]
majority = Counter(votes).most_common(1)[0][0] if votes else None

print("samples :", samples)
print("majority:", majority, "  golden:", HARD["golden_answer"])

A single sampled run stands or falls with one path; the vote marginalizes over paths.
The price is five calls instead of one — the accuracy-for-compute exchange named
test-time compute (notes Ch. 2), which Ch. 11's theory follows into the model's own training.

## 6. Completion Check

Completion criteria: the CoT prompt reaches the Section 3 target and the few-shot
prompt classifies every email. All rows must read `PASS` before submission; grading
checks these structural facts, never prose quality.

In [ ]:
completion = {
    "CoT prompt reaches >= 11/12 (Section 3.3)":  cot_score >= 11,
    "few-shot emails all pass (Section 4.2)":     email_score == 4,
}
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nLAB COMPLETE" if all(completion.values()) else "\nNOT COMPLETE YET")

---

Reference fill-ins: `labs/checkpoints/week02/solution.py` (lab and homework
together), published after the homework deadline.

W3 turns plain Python functions into tools the model can call (DeepLearning.AI,
*Agentic AI*, Module 3): function calling, the request–execute–reinject trace, and a
measured routing score — on the same `chat` helper built here.